# Weapon Detection - YOLO11n Fine-Tuning (Colab)

Trains a **YOLO11n** firearm/knife detector on the **Sohas** weapon dataset,
which deliberately includes *lookalike* objects (smartphone, wallet, banknote,
card) as their own classes. Learning those lookalikes is what stops the benign
false alarms that plagued the pretrained weapon models. You can also inject your
own **home hard-negative frames** as background images for an extra FP cut.

YOLO11n is the smallest variant - trains fast and runs comfortably on a 4 GB GPU.

Output checkpoint:  weapon_yolo11n_best.pt
The app's weapon stage already filters detections down to firearm/knife by class
name, so keeping the lookalike classes here is correct (they teach contrast).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =====================================================================
#  CONFIG  -  after you upload the whole Fine_tuning folder to
#  MyDrive/VisionAI/, these paths already point at this model folder.
#  (datasets live INSIDE it - see DATASETS.txt.)
# =====================================================================
BASE = "/content/drive/MyDrive/VisionAI/Fine_tuning/03_weapon_yolo"

# YOLO data.yaml of the weapon dataset (Roboflow/Kaggle export ships one).
DATA_YAML       = BASE + "/data.yaml"

# OPTIONAL: a folder of your own benign home FRAMES (.jpg/.png), NO labels.
# They get copied into the train split as background images -> fewer home FPs.
NEG_IMAGES_DIR  = BASE + "/home_negatives_frames"

WORK_DIR        = BASE + "/_work"

MODEL    = "yolo11n.pt"   # smallest; 4GB-friendly
EPOCHS   = 80
IMGSZ    = 640
BATCH    = -1             # -1 = auto-batch (~60% VRAM); pin e.g. 48 if you prefer
PATIENCE = 20             # early stop
SEED     = 42
N_SPLITS = 3              # k-fold CV (3 keeps total train time bounded)

import os
os.makedirs(WORK_DIR, exist_ok=True)
print("Work dir:", WORK_DIR)

In [ ]:
!pip -q install ultralytics pyyaml
import torch, subprocess
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print(subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"))

## Normalize data.yaml paths
Roboflow/Kaggle exports often ship `train: ../train/images` style paths that
break once the folder is on Drive. This rewrites `data.yaml` to absolute-safe
paths based on where it actually sits, so training Just Works.

In [ ]:
import os, yaml
ddir = os.path.dirname(DATA_YAML)
cfg = yaml.safe_load(open(DATA_YAML))
def find_split(*cands):
    for c in cands:
        p = os.path.join(ddir, c)
        if os.path.isdir(p): return p
    return None
train_dir = find_split('train/images', 'train')
val_dir   = find_split('valid/images', 'val/images', 'valid', 'val') or train_dir
test_dir  = find_split('test/images', 'test')
assert train_dir, "Could not find a train images folder next to data.yaml"
cfg['path']  = ddir
cfg['train'] = os.path.relpath(train_dir, ddir)
cfg['val']   = os.path.relpath(val_dir, ddir)
if test_dir: cfg['test'] = os.path.relpath(test_dir, ddir)
yaml.safe_dump(cfg, open(DATA_YAML, 'w'))
print("Classes (%d):" % cfg.get('nc', len(cfg.get('names', []))), cfg.get('names'))
print("train:", cfg['train'], "| val:", cfg['val'])

## (Optional) Inject home hard-negatives as background images
Ultralytics treats images with no label file as background. Adding benign home
frames here directly attacks the "phone/remote in living room -> firearm" FP.

In [ ]:
import os, glob, shutil, yaml
if NEG_IMAGES_DIR and os.path.isdir(NEG_IMAGES_DIR):
    cfg = yaml.safe_load(open(DATA_YAML))
    base = cfg.get("path", os.path.dirname(DATA_YAML))
    train_field = cfg.get("train", "train/images")
    train_images = train_field if os.path.isabs(train_field) else os.path.join(base, train_field)
    if not os.path.isdir(train_images):  # some exports point 'train' at the split dir
        train_images = os.path.join(train_images, "images") if os.path.isdir(os.path.join(train_images, "images")) else train_images
    os.makedirs(train_images, exist_ok=True)
    negs = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        negs += glob.glob(os.path.join(NEG_IMAGES_DIR, "**", ext), recursive=True)
    for i, p in enumerate(negs):
        shutil.copy(p, os.path.join(train_images, "neg_%05d%s" % (i, os.path.splitext(p)[1])))
    print("Injected %d background negatives into %s" % (len(negs), train_images))
else:
    print("No NEG_IMAGES_DIR (skipping). You can still train without it.")

## Cross-validated training (Stratified K-Fold)

Instead of one fixed split, we run **Stratified K-Fold** over every labeled
image - stratified by each image's dominant class so firearm / knife / lookalike
/ background stay balanced across folds. Each fold trains a fresh YOLO11n and
Ultralytics keeps that fold's `best.pt` (best validation mAP). The deployment
model is the fold with the highest mAP50-95.

**Speed:** the dataset splits are staged to local SSD (`/content/weapon_ds`) and
the fold trees + YOLO runs live under `/content/cv` - Drive is far too slow for
per-image reads. Only the final `weapon_yolo11n_best.pt` is written to `_work`
on Drive, so `_work` stays clean and re-runs never need manual cleanup
(`exist_ok=True`). `/content` is wiped when the runtime disconnects, so re-run
this cell after a reconnect to re-stage.

Note: k folds means k full trainings; `N_SPLITS=3` keeps total time bounded.

In [ ]:
import os, glob, shutil, json, time, yaml
import numpy as np
from sklearn.model_selection import StratifiedKFold
from ultralytics import YOLO

cfg = yaml.safe_load(open(DATA_YAML))
ddir_drive = cfg.get("path", os.path.dirname(DATA_YAML))
names = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][i] for i in sorted(cfg["names"])]
NC = len(names)

# ---- Stage dataset splits to local SSD once (Drive is too slow per-image) ----
LOCAL_DS = "/content/weapon_ds"
os.makedirs(LOCAL_DS, exist_ok=True)
for field in (cfg.get("train"), cfg.get("val"), cfg.get("test")):
    if not field: continue
    top = field.replace("\\", "/").split("/")[0]          # 'train' / 'valid' / 'test'
    src = os.path.join(ddir_drive, top); dst = os.path.join(LOCAL_DS, top)
    if os.path.isdir(src) and not os.path.isdir(dst):
        print("Staging %s -> local ..." % top); shutil.copytree(src, dst)
ddir = LOCAL_DS                                           # everything below reads local

def imgs_for(split_field):
    if not split_field: return []
    p = split_field if os.path.isabs(split_field) else os.path.join(ddir, split_field)
    if os.path.isdir(os.path.join(p, "images")): p = os.path.join(p, "images")
    out = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
        out += glob.glob(os.path.join(p, "**", ext), recursive=True)
    return out

all_imgs = []
for f in (cfg.get("train"), cfg.get("val"), cfg.get("test")):
    all_imgs += imgs_for(f)
all_imgs = sorted(set(all_imgs))

def label_path(img):
    d = os.path.dirname(img).replace(os.sep + "images", os.sep + "labels")
    return os.path.join(d, os.path.splitext(os.path.basename(img))[0] + ".txt")

def strat_label(img):
    lp = label_path(img)
    if not os.path.isfile(lp): return NC          # background / hard-negative
    cls = [int(float(ln.split()[0])) for ln in open(lp) if ln.strip()]
    return max(set(cls), key=cls.count) if cls else NC

all_imgs = np.array(all_imgs)
y_strat = np.array([strat_label(p) for p in all_imgs])
print("Images: %d | strat-class hist:" % len(all_imgs),
      {int(c): int((y_strat == c).sum()) for c in np.unique(y_strat)})

# ---- Fold trees + YOLO runs live on LOCAL disk (fast) ----
CV_ROOT = "/content/cv"
def build_fold_dir(fold, tr_idx, va_idx):
    root = os.path.join(CV_ROOT, "data", "fold%d" % fold)
    for sub in ("images/train", "images/val", "labels/train", "labels/val"):
        d = os.path.join(root, sub); os.makedirs(d, exist_ok=True)
        for f in glob.glob(os.path.join(d, "*")): os.remove(f)
    def link(idxs, split):
        for i in idxs:
            img = str(all_imgs[i]); stem = os.path.splitext(os.path.basename(img))[0]
            dst_img = os.path.join(root, "images", split, os.path.basename(img))
            try: os.symlink(img, dst_img)
            except (OSError, NotImplementedError): shutil.copy(img, dst_img)
            lp = label_path(img)
            if os.path.isfile(lp):
                dst_lab = os.path.join(root, "labels", split, stem + ".txt")
                try: os.symlink(lp, dst_lab)
                except (OSError, NotImplementedError): shutil.copy(lp, dst_lab)
    link(tr_idx, "train"); link(va_idx, "val")
    yml = os.path.join(root, "data.yaml")
    yaml.safe_dump({"path": root, "train": "images/train", "val": "images/val",
                    "nc": NC, "names": names}, open(yml, "w"))
    return yml

# ---- RESUME-SAFE: each finished fold's best.pt + result are saved to DRIVE.
#      Re-run after a disconnect -> completed folds are skipped automatically. ----
CV_SAVE = os.path.join(WORK_DIR, "cv_folds"); os.makedirs(CV_SAVE, exist_ok=True)
RESULTS_JSON = os.path.join(WORK_DIR, "cv_results.json")
results = json.load(open(RESULTS_JSON)) if os.path.isfile(RESULTS_JSON) else {}
if results:
    print("Resuming: %d/%d folds already done ->" % (len(results), N_SPLITS),
          {k: round(v["map"], 4) for k, v in results.items()})

os.makedirs(os.path.join(CV_ROOT, "runs"), exist_ok=True)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fold, (tr_idx, va_idx) in enumerate(skf.split(all_imgs, y_strat)):
    data_yaml = build_fold_dir(fold, tr_idx, va_idx)      # cheap symlinks; also needed by final eval
    saved = os.path.join(CV_SAVE, "fold%d_best.pt" % fold)
    if str(fold) in results and os.path.isfile(saved):
        results[str(fold)]["data"] = data_yaml            # refresh this session's local path
        print("Fold %d: already done (mAP50-95=%.4f) - skip" % (fold, results[str(fold)]["map"]))
        continue
    print("\n===== Fold %d/%d  train=%d val=%d =====" % (fold + 1, N_SPLITS, len(tr_idx), len(va_idx)))
    t0 = time.time()
    model = YOLO(MODEL)
    model.train(
        data=data_yaml, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        patience=PATIENCE, seed=SEED, project=os.path.join(CV_ROOT, "runs"),
        name="fold%d" % fold, exist_ok=True, pretrained=True, optimizer="auto", cos_lr=True,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, fliplr=0.5, mosaic=1.0,
        device=0, verbose=True)
    best_pt = os.path.join(CV_ROOT, "runs", "fold%d" % fold, "weights", "best.pt")
    vm = YOLO(best_pt).val(data=data_yaml, imgsz=IMGSZ, device=0, verbose=False)
    map95 = float(vm.box.map)
    shutil.copy(best_pt, saved)                           # persist fold to Drive immediately
    results[str(fold)] = {"fold": fold, "map": map95, "ckpt": saved, "data": data_yaml}
    json.dump(results, open(RESULTS_JSON, "w"))           # checkpoint progress to Drive
    print("Fold %d done in %.1f min | mAP50-95=%.4f -> %s" % (fold, (time.time() - t0) / 60, map95, saved))

fold_results = [results[str(f)] for f in range(N_SPLITS) if str(f) in results]
maps = [r["map"] for r in fold_results]
print("\nCV mAP50-95 per fold:", [round(v, 4) for v in maps],
      "| mean=%.4f" % (sum(maps) / len(maps)))
best = max(fold_results, key=lambda r: r["map"])
print("Selected fold %d (highest mAP50-95=%.4f)" % (best["fold"], best["map"]))


## Evaluate the selected fold + export the checkpoint
Validates the best-mAP fold on its own held-out fold split, prints mAP /
per-class, and copies it to `weapon_yolo11n_best.pt`.

In [ ]:
import shutil, os
from ultralytics import YOLO

# Deployment model = the CV fold with the highest mAP50-95.
sel_ckpt = best["ckpt"]
m = YOLO(sel_ckpt)
metrics = m.val(data=best["data"], imgsz=IMGSZ, device=0)
print("Selected fold %d | mAP50-95=%.4f" % (best["fold"], best["map"]))
print("mAP50-95:", round(float(metrics.box.map), 4), "| mAP50:", round(float(metrics.box.map50), 4))
try:
    for i, name in m.names.items():
        print("  %-14s mAP50=%.3f" % (name, float(metrics.box.maps[i])))
except Exception as e:
    print("per-class map unavailable:", e)

export_path = os.path.join(WORK_DIR, "weapon_yolo11n_best.pt")
shutil.copy(sel_ckpt, export_path)
print("Exported ->", export_path)


## Export & wire back
`weapon_yolo11n_best.pt` is in `WORK_DIR` on Drive.
1. Download it into the repo at `models/weights/weapon_yolo11n_best.pt`.
2. Point the weapon detector at it in `config.yaml`:
       models.weapon.detectors:
         - name: weapon_finetuned
           model_id: models/weights/weapon_yolo11n_best.pt
           filename: ""          # local path -> filename ignored
           confidence: 0.45      # tune from the val output
   The existing `stages/weapon_detection.py` already maps class names
   (pistol/gun/knife/...) to firearm/knife and drops the lookalike classes, so
   no code change is needed - just the config swap.
3. Keep the pose/proximity + temporal-persistence gating in the pipeline; with a
   domain-matched model + hard negatives it should finally hold at home.

**Tip:** if you still see benign FPs, add more of those exact scenes to
NEG_IMAGES_DIR and retrain - that is the highest-leverage fix.

## ADD-ON — run ONLY fold1 (fold0 already trained)

Run cells **1–7** first (mount, pip, normalize paths, optional negatives), then run the
**two cells below INSTEAD of** the original Train (cell 9) + Evaluate (cell 11).

- **Cell A** trains only fold1, fresh from `yolo11n.pt` (fold0 untouched, fold2 skipped),
  saves `fold1_best.pt` to Drive, prints the 2-fold CV mean.
- **Cell B** exports **fold0's `best.pt`** as the deployment model and prints the CV summary.


In [ ]:
# === ADD-ON CELL A : run ONLY fold1 (fold0 already done). SAME configs. ===
# Run cells 1-7 first (drive mount, pip, normalize paths, optional negatives),
# then SKIP the original training cell (cell 9) and run THIS instead. It is
# self-contained: re-stages to local SSD (cheap if already staged), rebuilds the
# fold index, then trains ONLY the folds in FOLDS_TO_RUN, saving each fold's
# best.pt to Drive the moment it finishes. fold1 trains FRESH from yolo11n.pt
# (NOT warm-started from fold0 - that would leak fold0's val data into fold1).
import os, glob, shutil, json, time, yaml
import numpy as np
from sklearn.model_selection import StratifiedKFold
from ultralytics import YOLO

FOLDS_TO_RUN = [1]        # fold0 already trained at 80 epochs; fold2 skipped
FOLD0_MAP    = 0.787      # used only for the CV mean, and only if fold0 isn't
                          # already recorded in cv_results.json

# ---- rebuild dataset index + fold builder (identical to the original cell) ----
cfg = yaml.safe_load(open(DATA_YAML))
ddir_drive = cfg.get("path", os.path.dirname(DATA_YAML))
names = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][i] for i in sorted(cfg["names"])]
NC = len(names)
LOCAL_DS = "/content/weapon_ds"; os.makedirs(LOCAL_DS, exist_ok=True)
for field in (cfg.get("train"), cfg.get("val"), cfg.get("test")):
    if not field: continue
    top = field.replace("\\", "/").split("/")[0]
    src = os.path.join(ddir_drive, top); dst = os.path.join(LOCAL_DS, top)
    if os.path.isdir(src) and not os.path.isdir(dst):
        print("Staging %s -> local ..." % top); shutil.copytree(src, dst)
ddir = LOCAL_DS

def imgs_for(split_field):
    if not split_field: return []
    p = split_field if os.path.isabs(split_field) else os.path.join(ddir, split_field)
    if os.path.isdir(os.path.join(p, "images")): p = os.path.join(p, "images")
    out = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
        out += glob.glob(os.path.join(p, "**", ext), recursive=True)
    return out

all_imgs = []
for f in (cfg.get("train"), cfg.get("val"), cfg.get("test")):
    all_imgs += imgs_for(f)
all_imgs = sorted(set(all_imgs))

def label_path(img):
    d = os.path.dirname(img).replace(os.sep + "images", os.sep + "labels")
    return os.path.join(d, os.path.splitext(os.path.basename(img))[0] + ".txt")

def strat_label(img):
    lp = label_path(img)
    if not os.path.isfile(lp): return NC
    cls = [int(float(ln.split()[0])) for ln in open(lp) if ln.strip()]
    return max(set(cls), key=cls.count) if cls else NC

all_imgs = np.array(all_imgs)
y_strat = np.array([strat_label(p) for p in all_imgs])
print("Images: %d" % len(all_imgs))

CV_ROOT = "/content/cv"
def build_fold_dir(fold, tr_idx, va_idx):
    root = os.path.join(CV_ROOT, "data", "fold%d" % fold)
    for sub in ("images/train", "images/val", "labels/train", "labels/val"):
        d = os.path.join(root, sub); os.makedirs(d, exist_ok=True)
        for f in glob.glob(os.path.join(d, "*")): os.remove(f)
    def link(idxs, split):
        for i in idxs:
            img = str(all_imgs[i]); stem = os.path.splitext(os.path.basename(img))[0]
            dst_img = os.path.join(root, "images", split, os.path.basename(img))
            try: os.symlink(img, dst_img)
            except (OSError, NotImplementedError): shutil.copy(img, dst_img)
            lp = label_path(img)
            if os.path.isfile(lp):
                dst_lab = os.path.join(root, "labels", split, stem + ".txt")
                try: os.symlink(lp, dst_lab)
                except (OSError, NotImplementedError): shutil.copy(lp, dst_lab)
    link(tr_idx, "train"); link(va_idx, "val")
    yml = os.path.join(root, "data.yaml")
    yaml.safe_dump({"path": root, "train": "images/train", "val": "images/val",
                    "nc": NC, "names": names}, open(yml, "w"))
    return yml

# ---- train only the requested folds; persist each best.pt to Drive ----
CV_SAVE = os.path.join(WORK_DIR, "cv_folds"); os.makedirs(CV_SAVE, exist_ok=True)
RESULTS_JSON = os.path.join(WORK_DIR, "cv_results.json")
results = json.load(open(RESULTS_JSON)) if os.path.isfile(RESULTS_JSON) else {}
results.setdefault("0", {"fold": 0, "map": FOLD0_MAP, "ckpt": None, "data": None})
os.makedirs(os.path.join(CV_ROOT, "runs"), exist_ok=True)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fold, (tr_idx, va_idx) in enumerate(skf.split(all_imgs, y_strat)):
    if fold not in FOLDS_TO_RUN:
        continue
    data_yaml = build_fold_dir(fold, tr_idx, va_idx)
    saved = os.path.join(CV_SAVE, "fold%d_best.pt" % fold)
    if str(fold) in results and results[str(fold)].get("ckpt") and os.path.isfile(saved):
        results[str(fold)]["data"] = data_yaml
        print("Fold %d already done (mAP=%.4f) - skip" % (fold, results[str(fold)]["map"])); continue
    print("\n===== Fold %d/%d  train=%d val=%d =====" % (fold + 1, N_SPLITS, len(tr_idx), len(va_idx)))
    t0 = time.time()
    model = YOLO(MODEL)
    model.train(
        data=data_yaml, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        patience=PATIENCE, seed=SEED, project=os.path.join(CV_ROOT, "runs"),
        name="fold%d" % fold, exist_ok=True, pretrained=True, optimizer="auto", cos_lr=True,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, fliplr=0.5, mosaic=1.0,
        device=0, verbose=True)
    best_pt = os.path.join(CV_ROOT, "runs", "fold%d" % fold, "weights", "best.pt")
    vm = YOLO(best_pt).val(data=data_yaml, imgsz=IMGSZ, device=0, verbose=False)
    map95 = float(vm.box.map)
    shutil.copy(best_pt, saved)
    results[str(fold)] = {"fold": fold, "map": map95, "ckpt": saved, "data": data_yaml}
    json.dump(results, open(RESULTS_JSON, "w"))
    print("Fold %d done in %.1f min | mAP50-95=%.4f -> %s" % (fold, (time.time()-t0)/60, map95, saved))

done = [results[str(f)] for f in range(N_SPLITS) if str(f) in results]
maps = [r["map"] for r in done]
print("\nCV mAP50-95 per fold:", {r["fold"]: round(r["map"], 4) for r in done},
      "| mean=%.4f +/- %.4f" % (float(np.mean(maps)), float(np.std(maps))))


In [ ]:
# === ADD-ON CELL B : CV summary + export fold0 as the deployment model ===
import os, json, shutil
import numpy as np
from ultralytics import YOLO

# Deploy fold0 (confirmed: best.pt + last.pt exist in cv_folds/fold0_weights/).
# Set to None to instead auto-pick the highest-mAP fold that has a saved ckpt.
DEPLOY_CKPT_OVERRIDE = WORK_DIR + "/cv_folds/fold0_weights/best.pt"

RESULTS_JSON = os.path.join(WORK_DIR, "cv_results.json")
results = json.load(open(RESULTS_JSON))
done = [results[k] for k in sorted(results)]
maps = [r["map"] for r in done]
print("CV mAP50-95 per fold:", {r["fold"]: round(r["map"], 4) for r in done})
print("CV mean=%.4f  std=%.4f  (n=%d)" % (float(np.mean(maps)), float(np.std(maps)), len(maps)))

have_ckpt = [r for r in done if r.get("ckpt") and os.path.isfile(r["ckpt"])]
sel = None
if DEPLOY_CKPT_OVERRIDE and os.path.isfile(DEPLOY_CKPT_OVERRIDE):
    sel_ckpt = DEPLOY_CKPT_OVERRIDE
    print("Deploying override ckpt:", sel_ckpt)
elif have_ckpt:
    sel = max(have_ckpt, key=lambda r: r["map"]); sel_ckpt = sel["ckpt"]
    print("Deploying highest-mAP fold with saved weights: fold %d (mAP=%.4f)" % (sel["fold"], sel["map"]))
else:
    raise SystemExit("No saved checkpoint found and override path missing.")

m = YOLO(sel_ckpt)
if sel and sel.get("data") and os.path.isfile(sel["data"]):
    metrics = m.val(data=sel["data"], imgsz=IMGSZ, device=0)
    print("mAP50-95:", round(float(metrics.box.map), 4), "| mAP50:", round(float(metrics.box.map50), 4))
    try:
        for i, name in m.names.items():
            print("  %-14s mAP50=%.3f" % (name, float(metrics.box.maps[i])))
    except Exception as e:
        print("per-class map unavailable:", e)

export_path = os.path.join(WORK_DIR, "weapon_yolo11n_best.pt")
shutil.copy(sel_ckpt, export_path)
print("Exported ->", export_path)
